# Lineborn v6 — One-click deployment build

This notebook turns the trained **Lineborn v6 DPO LoRA** into the two local shipping candidates:

- **Balanced:** `lineborn-v6-balanced` → `Q4_K_M`
- **Performance:** `lineborn-v6-performance` → `Q8_0`

It automatically finds the eight adapter ZIP parts in Google Drive, reconstructs and verifies the exact archive, merges the final DPO adapter into **Qwen/Qwen3-4B-Instruct-2507**, converts it to GGUF, quantizes both builds, and saves the finished artifacts back to Google Drive.

**Use a GPU runtime.** In Colab: **Runtime → Change runtime type → T4 GPU**, then choose **Runtime → Run all**.

This notebook does **not** replace the shipping Lineborn runtime. The generated models still have to pass the strict release benchmark.


In [ ]:
# 1) Runtime preflight
import os, shutil, subprocess, sys
from pathlib import Path

print('Python:', sys.version.split()[0])
try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError('GPU runtime not detected. In Colab select Runtime > Change runtime type > T4 GPU.') from exc

total, used, free = shutil.disk_usage('/content')
print(f'Local disk free: {free/1024**3:.1f} GiB')
if free < 30 * 1024**3:
    raise RuntimeError('At least ~30 GiB of free Colab disk is recommended for merge + F16 + Q4 + Q8 artifacts.')
print('Preflight OK.')


In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3) Configuration
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
OUTPUT_DRIVE = DRIVE_ROOT / 'Lineborn-v6-export'
REPO_DIR = Path('/content/lineborn-runtime')
ARCHIVE = Path('/content/lineborn-sales-adapters-v6.zip')
EXTRACT_DIR = Path('/content/lineborn-v6-adapters')
EXPORT_DIR = Path('/content/lineborn-v6-export')

EXPECTED_ARCHIVE_BYTES = 1_800_343_183
EXPECTED_ARCHIVE_SHA256 = 'ea657b97036665d0e9cbd95b09523f0a224ce40485b3fc9e02361b816e09f406'
EXPECTED_PART_BYTES = {i: 225_042_898 for i in range(1, 8)} | {8: 225_042_897}

REPO = 'https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git'
BRANCH = 'lineborn-v6-deployment'
BASE_MODEL = 'Qwen/Qwen3-4B-Instruct-2507'

OUTPUT_DRIVE.mkdir(parents=True, exist_ok=True)
print('Drive output:', OUTPUT_DRIVE)


In [ ]:
# 4) Find the exact 8 split files anywhere in My Drive
parts = []
for i in range(1, 9):
    name = f'lineborn-sales-adapters-v6.zip.part{i}'
    matches = [p for p in DRIVE_ROOT.rglob(name) if p.is_file() and p.stat().st_size == EXPECTED_PART_BYTES[i]]
    if not matches:
        raise FileNotFoundError(f'Could not find {name} with expected size {EXPECTED_PART_BYTES[i]:,} bytes under {DRIVE_ROOT}')
    if len(matches) > 1:
        print(f'Warning: multiple valid-size matches for {name}; using newest:')
        matches.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        for candidate in matches:
            print(' ', candidate)
    parts.append(matches[0])

print('Found all parts:')
for p in parts:
    print(f'  {p.name}: {p.stat().st_size:,} bytes — {p}')


In [ ]:
# 5) Reconstruct the archive and verify its exact SHA-256
import hashlib, time

if ARCHIVE.exists():
    ARCHIVE.unlink()

h = hashlib.sha256()
written = 0
started = time.time()
with ARCHIVE.open('wb') as dst:
    for idx, part in enumerate(parts, 1):
        print(f'Joining part {idx}/8: {part.name}')
        with part.open('rb') as src:
            while True:
                chunk = src.read(16 * 1024 * 1024)
                if not chunk:
                    break
                dst.write(chunk)
                h.update(chunk)
                written += len(chunk)
        print(f'  total written: {written/1024**3:.3f} GiB')

sha = h.hexdigest()
print('Archive bytes:', written)
print('Archive SHA256:', sha)
print(f'Elapsed: {time.time()-started:.1f}s')
assert written == EXPECTED_ARCHIVE_BYTES, (written, EXPECTED_ARCHIVE_BYTES)
assert sha == EXPECTED_ARCHIVE_SHA256, (sha, EXPECTED_ARCHIVE_SHA256)
print('✅ Exact v6 archive verified.')


In [ ]:
# 6) Clone the deployment branch and install dependencies
import os, subprocess

subprocess.run(['rm','-rf',str(REPO_DIR)], check=True)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable,'-m','pip','install','-q','-r','training/requirements-colab.txt'], check=True)
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','ninja-build'], check=True)
print('✅ Repository and build dependencies ready.')


In [ ]:
# 7) Verify manifest/file hashes and extract the trained adapters
import subprocess, shutil

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
subprocess.run([
    sys.executable,
    'training/verify_lineborn_v6_export.py',
    str(ARCHIVE),
    '--extract-dir', str(EXTRACT_DIR),
], check=True)

adapter = EXTRACT_DIR / 'lineborn-sales-dpo' / 'adapter'
assert (adapter/'adapter_config.json').is_file()
assert (adapter/'adapter_model.safetensors').is_file()
print('✅ Final DPO adapter:', adapter)


In [ ]:
# 8) Merge + GGUF conversion + quantization
# The exporter uses the GPU-aware merge script, pins llama.cpp to the validated commit,
# and generates both Balanced Q4_K_M and Performance Q8_0 builds.
import os, subprocess, shutil

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

env = os.environ.copy()
env['LINEBORN_BASE_MODEL'] = BASE_MODEL
env['LINEBORN_BALANCED_QUANT'] = 'Q4_K_M'
env['LINEBORN_PERFORMANCE_QUANT'] = 'Q8_0'
env['LINEBORN_KEEP_F16'] = '0'

subprocess.run([
    'bash', 'training/export_lineborn_v6_gguf.sh',
    str(adapter), str(EXPORT_DIR)
], check=True, env=env)
print('✅ Merge and quantization complete.')


In [ ]:
# 9) Validate generated artifacts
import json, hashlib

manifest_path = EXPORT_DIR / 'lineborn-v6-deployment-manifest.json'
manifest = json.loads(manifest_path.read_text())
assert manifest['base_model'] == BASE_MODEL

required = [
    'lineborn-v6-balanced-q4_k_m.gguf',
    'lineborn-v6-performance-q8_0.gguf',
    'Modelfile.balanced',
    'Modelfile.performance',
    'lineborn-v6-deployment-manifest.json',
]
for name in required:
    p = EXPORT_DIR / name
    assert p.is_file(), f'Missing {p}'
    print(f'{name}: {p.stat().st_size/1024**3:.3f} GiB' if p.suffix == '.gguf' else f'{name}: {p.stat().st_size:,} bytes')

print('\nManifest:')
print(json.dumps(manifest, indent=2))
print('\n✅ Candidate artifacts validated.')


In [ ]:
# 10) Save the final candidates to Google Drive
import shutil, time

OUTPUT_DRIVE.mkdir(parents=True, exist_ok=True)
for name in required:
    src = EXPORT_DIR / name
    dst = OUTPUT_DRIVE / name
    print('Copying', name, '→ Drive...')
    shutil.copy2(src, dst)
    assert dst.stat().st_size == src.stat().st_size

# Also save provenance for the exact adapter archive used.
provenance = OUTPUT_DRIVE / 'lineborn-v6-source.txt'
provenance.write_text(
    f'base_model={BASE_MODEL}\n'
    f'adapter_archive_sha256={EXPECTED_ARCHIVE_SHA256}\n'
    f'adapter_archive_bytes={EXPECTED_ARCHIVE_BYTES}\n'
    f'repo_branch={BRANCH}\n',
    encoding='utf-8'
)
print('\n✅ Finished. Saved to:', OUTPUT_DRIVE)
print('Balanced:', OUTPUT_DRIVE / 'lineborn-v6-balanced-q4_k_m.gguf')
print('Performance:', OUTPUT_DRIVE / 'lineborn-v6-performance-q8_0.gguf')


## After this finishes

Your Drive folder `MyDrive/Lineborn-v6-export/` will contain the two trained GGUF candidates plus their Ollama Modelfiles and deployment manifest.

Next we import them into Ollama as:

```text
lineborn-v6-balanced
lineborn-v6-performance
```

and run `benchmarks/lineborn_trained_release_acceptance_v6.py` against both. **Do not switch the shipping runtime until both pass with zero critical failures and zero control leaks.**
